# 🛡️ ZENTRA — Person Detection Training (Colab, ฟรี T4 GPU)

เทรน **person detector** (รากฐานของทั้งระบบ — PPE / Zone / Fall ยืนบนกล่องคนทั้งหมด)
บน Google Colab GPU ฟรี แล้วเอา `.pt` กลับไปวางใน `backend/models/` ของเครื่องคุณ

> **ทำไมต้องเทรนที่นี่:** เครื่องคุณเป็น CPU-only (i5-1135G7) → เทรน yolo11s ~2 วัน
> บน Colab T4 ฟรี ~30 นาที (เร็วกว่า ~50-100 เท่า)

### วิธีใช้
1. เมนู **Runtime → Change runtime type → Hardware accelerator = T4 GPU** → Save
2. กด Run ทีละเซลล์จากบนลงล่าง (Shift+Enter)
3. เซลล์สุดท้ายจะดาวน์โหลด `person_v1.pt` กลับเครื่องคุณ

### เป้าหมาย (Gate 1 ของโปรเจกต์)
**mAP50 ≥ 0.75** ที่ imgsz 960 (ค่าที่ deploy จริง) · recall ต้องดีขึ้นจาก baseline COCO 0.623


## 1) ตรวจ GPU + ติดตั้ง ultralytics

In [ ]:
import subprocess, torch
print('CUDA available :', torch.cuda.is_available())
print('GPU            :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — ไปตั้ง Runtime→T4 GPU ก่อน!')
assert torch.cuda.is_available(), 'ยังไม่ได้เปิด GPU: Runtime → Change runtime type → T4 GPU'
!pip -q install ultralytics
import ultralytics; ultralytics.__version__


## 2) เตรียม Dataset — **รันเซลล์ 1 อันด้านล่างก่อนไปเทรน**

> ⚠️ ถ้าข้ามหัวข้อนี้ไปเทรนเลย จะเจอ `NameError: DATA_YAML is not defined`
> เพราะยังไม่ได้เลือกข้อมูล — เลือก **1 วิธี** แล้วรันให้ `DATA_YAML` ถูกตั้งค่าก่อน

- **Quick start (ทดสอบว่ารันได้ — zero config):** โหลด `coco128` อัตโนมัติ 128 ภาพ
  แค่พิสูจน์ว่า pipeline ทำงาน (ยังไม่ใช่โมเดลจริง — ข้อมูลน้อยเกิน)
- **วิธี A — Roboflow Universe (แนะนำสำหรับโมเดลจริง):** dataset คน/CrowdHuman พร้อมใช้
  ดาวน์โหลดบรรทัดเดียว ต้องมี **free API key** (roboflow.com → Settings → API key)
- **วิธี B — ข้อมูลของคุณเอง:** อัปโหลด zip (YOLO format มี data.yaml)

> **สำคัญ (บทเรียนของโปรเจกต์):** ใช้ dataset ที่แบ่ง train/valid/test **ตามภาพต้นฉบับ**
> ไม่ใช่ตามภาพ augment — ไม่งั้น mAP จะสูงเกินจริง (augmentation leakage)


### Quick start — coco128 (auto, zero config) — รันเพื่อทดสอบว่า notebook ทำงาน

In [ ]:
# ultralytics ดาวน์โหลด coco128 ให้อัตโนมัติ (128 ภาพ, มีคลาส person)
# ใช้แค่พิสูจน์ว่า train/eval/export ทำงานครบ — ข้อมูลน้อยเกินจะได้โมเดลจริง
DATA_YAML = 'coco128.yaml'
print('DATA_YAML =', DATA_YAML, '(smoke test — สลับไปวิธี A เพื่อโมเดลจริง)')


### วิธี A — Roboflow Universe (วางโค้ดที่ Roboflow สร้างให้ — การันตีใช้ได้)

**อย่าเดา slug เอง** (workspace ID ที่ API ใช้มักไม่ตรงกับ URL) — ให้ Roboflow สร้างโค้ดให้:

1. เปิดหน้า dataset คน เช่น
   [people-detection (leo-ueno)](https://universe.roboflow.com/leo-ueno/people-detection-o4rdr) หรือ
   [crowdhuman (keio)](https://universe.roboflow.com/keio-dba-team/crowdhuman-nur7g)
2. กดปุ่ม **Download Dataset** → Format = **YOLOv11** → เลือก **show download code**
3. Roboflow จะโชว์โค้ด 4-5 บรรทัด (มี `rf.workspace(...).project(...).version(...)` ที่ถูกต้อง + key ของคุณ)
4. **ก๊อปทั้งหมดมาวางแทน** ในเซลล์ล่าง (ตรงส่วน `<<< วาง >>>`)


In [ ]:
!pip -q install roboflow
from roboflow import Roboflow

# ========== วางโค้ดที่ Roboflow ให้มาแทน 3 บรรทัดนี้ ==========
rf = Roboflow(api_key='ใส่_API_KEY_ของคุณ')
project = rf.workspace('<<< วางจาก Roboflow >>>').project('<<< วางจาก Roboflow >>>')
dataset = project.version(1).download('yolov11')
# ============================================================

DATA_YAML = dataset.location + '/data.yaml'
print('✅ DATA_YAML =', DATA_YAML)


### วิธี B — อัปโหลดข้อมูลของคุณเอง (zip: YOLO format มี data.yaml)

In [ ]:
# รันเซลล์นี้ถ้าจะใช้ข้อมูลของคุณเอง แล้วเลือกไฟล์ zip ตอนถูกถาม
# โครงสร้าง zip ที่ถูกต้อง: train/images, train/labels, valid/images, valid/labels, data.yaml
USE_UPLOAD = False   # เปลี่ยนเป็น True เพื่อใช้เซลล์นี้
if USE_UPLOAD:
    from google.colab import files
    import zipfile, os, glob
    up = files.upload()
    zname = list(up.keys())[0]
    os.makedirs('/content/mydata', exist_ok=True)
    with zipfile.ZipFile(zname) as z: z.extractall('/content/mydata')
    hits = glob.glob('/content/mydata/**/data.yaml', recursive=True)
    DATA_YAML = hits[0] if hits else None
    print('DATA_YAML =', DATA_YAML)


## 3) เทรน yolo11s (hyperparameters เดียวกับ `scripts/train_person.py` ของโปรเจกต์)

บน T4 ~30-40 นาที สำหรับ ~3-4 พันภาพ 40 epochs


In [ ]:
from ultralytics import YOLO
import time

# guard: ต้องเลือก dataset ในหัวข้อ 2 ก่อน
try:
    DATA_YAML
except NameError:
    raise RuntimeError('ยังไม่ได้เลือก dataset — กลับไปรันเซลล์ในหัวข้อ 2 '
                       '(Quick start / วิธี A / วิธี B) ให้ DATA_YAML ถูกตั้งค่าก่อน')
assert DATA_YAML, 'DATA_YAML ว่างเปล่า — ตรวจว่าเซลล์ dataset รันสำเร็จ'
print('ใช้ dataset:', DATA_YAML)

# base = yolo11s (แนะนำสำหรับ person: recall คนไกลในฝูงชนดีกว่า n)
model = YOLO('yolo11s.pt')
t0 = time.time()
model.train(
    data      = DATA_YAML,
    epochs    = 40,
    imgsz     = 640,            # เทรนที่ 640 (ประเมินทั้ง 640 และ 960 ด้านล่าง)
    batch     = 16,             # T4 16GB ไหว; ลดเหลือ 8 ถ้า OOM
    device    = 0,
    workers   = 4,
    patience  = 10,             # early stop ถ้าไม่ดีขึ้น 10 epochs
    optimizer = 'auto', cos_lr = True, seed = 0, pretrained = True,
    # augmentation (ตรงกับโปรเจกต์)
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, degrees=0.0, translate=0.1,
    scale=0.5, fliplr=0.5, mosaic=1.0, close_mosaic=10,
    project='/content/runs', name='person_v1', exist_ok=True,
    plots=True, val=True,
)
print(f'\nเทรนเสร็จใน {(time.time()-t0)/60:.1f} นาที')
BEST = '/content/runs/person_v1/weights/best.pt'
print('best.pt =', BEST)


## 4) วัดผลอย่างซื่อสัตย์ + เช็ค Gate 1

ประเมินที่ **ทั้ง imgsz 640 (เทรน) และ 960 (deploy จริง)** — สองค่านี้ให้ผลต่างกัน
และแตะ test split ครั้งเดียวตอนจบ (ไม่ใช้เลือกอะไร) — ตามหลักการของโปรเจกต์


In [ ]:
from ultralytics import YOLO
ft = YOLO(BEST)
base = YOLO('yolo11s.pt')   # COCO baseline (person class = 0)

def ev(m, tag, split, imgsz, **kw):
    r = m.val(data=DATA_YAML, split=split, imgsz=imgsz, verbose=False, plots=False, **kw)
    print(f'{tag:<26} {split:<5} @{imgsz}  mAP50={r.box.map50:.4f}  '
          f'mAP50-95={r.box.map:.4f}  P={r.box.mp:.4f}  R={r.box.mr:.4f}')
    return r.box.map50

print('===== Person Stage 1 results =====')
ev(base, 'baseline COCO yolo11s', 'val', 640, classes=[0])
ev(ft,   'fine-tuned person_v1',  'val', 640)
g = ev(ft, 'fine-tuned person_v1', 'val', 960)          # <-- Gate ที่ imgsz deploy
try:
    ev(ft, 'fine-tuned person_v1', 'test', 960)         # แตะครั้งเดียว
except Exception as e:
    print('(ไม่มี test split — ข้าม)')

print(f"\nGate 1 (mAP50 >= 0.75 @ imgsz 960): {'PASS ✅' if g>=0.75 else 'FAIL ❌'} — {g:.4f}")
if g < 0.75:
    print('ยังไม่ผ่าน → เพิ่ม epochs / เพิ่มข้อมูล / ตรวจ label ก่อน deploy')


## 5) ดาวน์โหลดโมเดลกลับเครื่อง → วางใน ZENTRA

**วิธีที่ง่ายและปลอดภัยสุด (แนะนำ):** เขียนทับไฟล์ person model เดิม
1. ดาวน์โหลด `person_v1.pt` (เซลล์ถัดไป)
2. เปลี่ยนชื่อเป็น `yolo11s.pt` แล้ววางทับที่ `backend/models/yolo11s.pt`
3. รันแอปใหม่ → ใช้โมเดลที่ fine-tune แล้วทันที (ไม่ต้องแก้ config)

**วิธีทางเลือก (เก็บชื่อแยก):** วาง `person_v1.pt` ไว้ที่ `backend/models/person_v1.pt`
แล้วเปิด `backend/.env` เพิ่ม `PERSON_MODEL=` เป็น **path เต็ม** ของไฟล์ เช่น
`PERSON_MODEL=D:\NSC26_Zentra\Application\ZENTRA_application-main\backend\models\person_v1.pt`
(ใช้ path เต็มเพราะ path แบบ relative อาจ resolve ผิดตาม cwd → โมเดลไม่โหลด)


In [ ]:
import shutil
shutil.copy(BEST, '/content/person_v1.pt')
from google.colab import files
files.download('/content/person_v1.pt')
print('กำลังดาวน์โหลด person_v1.pt ... เอาไปวางใน backend/models/')
